# Job Opportunity Hunter — Multi-Agent System

### Valentine Awe — Week 8 Community Contribution

An autonomous agent framework that scans RSS feeds for tech jobs, matches them against your resume using RAG, estimates salary fairness, and notifies you of the best match.

```
ScannerAgent  →  PlanningAgent (GPT tool-calling)  →  ResumeRAGAgent (ChromaDB)
                         │                              SalaryEstimator (GPT)
                         └──→ MessagingAgent (notify best match)
```

In [ ]:
import os, json, re, logging, queue, threading, time
from typing import List, Optional

import feedparser
import numpy as np
import gradio as gr
import requests as http_requests
from pydantic import BaseModel
from openai import OpenAI
from sentence_transformers import SentenceTransformer
import chromadb
from dotenv import load_dotenv

load_dotenv(override=True)
openai_client = OpenAI()
logging.basicConfig(level=logging.INFO, format='[%(asctime)s] %(message)s', datefmt='%H:%M:%S')

## Data Models & Base Agent

In [ ]:
class JobListing(BaseModel):
    title: str
    company: str
    description: str
    salary_text: Optional[str] = None
    url: str

class JobSelection(BaseModel):
    jobs: List[JobListing]

class JobOpportunity(BaseModel):
    job: JobListing
    match_score: float = 0.0
    estimated_salary: float = 0.0
    overall_score: float = 0.0

    def summary(self) -> str:
        return f"{self.job.title} at {self.job.company} | Match: {self.match_score:.0f}% | ~${self.estimated_salary:,.0f} | Score: {self.overall_score:.1f}"


class Agent:
    name = "Agent"
    color = '\033[37m'
    RESET = '\033[0m'

    def log(self, message):
        logging.info(f"[{self.name}] {message}")

## Sample Resume

Replace this with your own for personalized matching.

In [ ]:
SAMPLE_RESUME = """
Valentine Awe — Software & AI Engineer

SKILLS: Python, JavaScript, TypeScript, React, Node.js, FastAPI, Django,
Machine Learning, LLMs, Fine-tuning, RAG, Prompt Engineering,
OpenAI API, LangChain, ChromaDB, HuggingFace, PostgreSQL, MongoDB,
Docker, Kubernetes, AWS, GCP, Git, CI/CD, REST APIs

EXPERIENCE:
- Built production LLM applications including RAG pipelines and multi-agent systems
- Full-stack web apps with React frontends and Python backends
- ML pipelines for data processing, training, and inference
- Cloud infrastructure and serverless deployment (Modal, AWS Lambda)

EDUCATION: Computer Science / Software Engineering
LLM Engineering bootcamp — fine-tuning, RAG, multi-agent systems, deployment
"""

## The Agents

In [ ]:
# Test data so we can run the full pipeline without network calls

FAKE_JOBS = [
    {"title": "Senior Python Engineer", "company": "Stripe",
     "description": "Build payment APIs using Python, FastAPI, PostgreSQL. Distributed systems and cloud experience required.",
     "salary_text": "$180K-$220K", "url": "https://stripe.com/jobs/123"},
    {"title": "ML Engineer", "company": "Shopify",
     "description": "Deploy ML models for recommendations. Python, PyTorch, MLOps. LLM and RAG experience preferred.",
     "salary_text": "$160K-$200K", "url": "https://shopify.com/jobs/456"},
    {"title": "Frontend Developer", "company": "Figma",
     "description": "Build design tools with React, TypeScript, WebGL. Strong CSS skills. No backend or ML needed.",
     "salary_text": "$150K-$180K", "url": "https://figma.com/jobs/789"},
    {"title": "AI Engineer", "company": "Anthropic",
     "description": "LLM deployment, fine-tuning, RAG systems, agent frameworks. Deep Python and transformer expertise required.",
     "salary_text": "$200K-$280K", "url": "https://anthropic.com/jobs/101"},
    {"title": "DevOps Engineer", "company": "Datadog",
     "description": "Kubernetes, CI/CD, Terraform, Docker, AWS. No ML experience needed.",
     "salary_text": "$155K-$190K", "url": "https://datadog.com/jobs/202"},
]

In [ ]:
class ScannerAgent(Agent):
    """Scrapes RSS feeds for job listings and uses GPT to extract structured data."""
    name = "Scanner"

    RSS_FEEDS = [
        "https://hnrss.org/whoishiring/jobs",
        "https://news.google.com/rss/search?q=software+engineer+hiring+remote&hl=en-US&gl=US&ceid=US:en",
    ]

    def scan(self) -> JobSelection:
        self.log("Scanning RSS feeds...")
        raw = []
        for url in self.RSS_FEEDS:
            try:
                feed = feedparser.parse(url)
                for e in feed.entries[:15]:
                    raw.append(f"Title: {e.get('title','N/A')}\nLink: {e.get('link','N/A')}\nSummary: {e.get('summary','N/A')[:400]}")
            except Exception as ex:
                self.log(f"Feed error: {ex}")

        self.log(f"Found {len(raw)} raw entries, extracting top 5...")
        response = openai_client.chat.completions.parse(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Extract the 5 best tech job listings. Return JSON matching the schema. Rephrase descriptions as concise role summaries. Set salary_text to null if unknown."},
                {"role": "user", "content": "\n\n---\n\n".join(raw)}
            ],
            response_format=JobSelection,
        )
        result = response.choices[0].message.parsed
        self.log(f"Extracted {len(result.jobs)} jobs")
        return result

    def test_scan(self) -> JobSelection:
        self.log("Returning test data (no network calls)")
        return JobSelection(jobs=[JobListing(**j) for j in FAKE_JOBS])

In [ ]:
class ResumeRAGAgent(Agent):
    """Embeds resume in ChromaDB, computes semantic match score vs job descriptions."""
    name = "Resume RAG"

    def __init__(self, resume_text: str):
        self.encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
        sections = [s.strip() for s in resume_text.strip().split('\n\n') if len(s.strip()) > 20]
        client = chromadb.Client()
        self.collection = client.get_or_create_collection("resume")
        if self.collection.count() == 0:
            vecs = self.encoder.encode(sections).astype(float).tolist()
            self.collection.add(ids=[f"s{i}" for i in range(len(sections))], documents=sections, embeddings=vecs)
        self.log(f"Ready — {len(sections)} resume sections embedded")

    def match(self, job_description: str) -> dict:
        vec = self.encoder.encode([job_description]).astype(float).tolist()
        results = self.collection.query(query_embeddings=vec, n_results=3)
        avg_dist = np.mean(results['distances'][0])
        score = max(0, min(100, (1 - avg_dist / 2) * 100))
        self.log(f"Match: {score:.1f}% for '{job_description[:50]}...'")
        return {"match_score": round(score, 1)}

In [ ]:
class SalaryEstimatorAgent(Agent):
    """Uses a frontier LLM to estimate fair market salary."""
    name = "Salary Estimator"

    def estimate(self, job_title: str, company: str) -> dict:
        self.log(f"Estimating salary: {job_title} at {company}")
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": 'Estimate fair US base salary. Respond ONLY with JSON: {"estimated_salary": <number>}'},
                {"role": "user", "content": f"{job_title} at {company}"}
            ],
            temperature=0.3,
        )
        raw = response.choices[0].message.content.strip()
        try:
            if '```' in raw:
                raw = raw.split('```')[1].lstrip('json')
            result = json.loads(raw)
        except json.JSONDecodeError:
            m = re.search(r'\d[\d,]+', raw)
            result = {"estimated_salary": float(m.group().replace(',','')) if m else 100000}
        self.log(f"Estimated ${result['estimated_salary']:,.0f}")
        return result

In [ ]:
class MessagingAgent(Agent):
    """Sends push notifications (Pushover) or prints to console."""
    name = "Messenger"

    def __init__(self):
        self.pushover_user = os.getenv("PUSHOVER_USER")
        self.pushover_token = os.getenv("PUSHOVER_TOKEN")
        self.log("Pushover configured" if self.pushover_user else "No Pushover — will print notifications")

    def alert(self, title: str, company: str, match_score: float, estimated_salary: float, url: str) -> str:
        msg = f"Job Match!\n{title} at {company}\nMatch: {match_score:.0f}% | ~${estimated_salary:,.0f}\n{url}"
        if self.pushover_user and self.pushover_token:
            http_requests.post("https://api.pushover.net/1/messages.json",
                data={"user": self.pushover_user, "token": self.pushover_token, "message": msg})
        self.log(f"NOTIFY: {title} at {company} — {match_score:.0f}% match")
        return "sent"

## Quick Agent Test

Verify each agent works before wiring them into the planner.

In [ ]:
scanner = ScannerAgent()
rag_agent = ResumeRAGAgent(SAMPLE_RESUME)
salary_agent = SalaryEstimatorAgent()
messenger = MessagingAgent()

# Test RAG matching — AI role should score higher than pure frontend role
print("AI Engineer match:", rag_agent.match("LLM deployment, fine-tuning, RAG systems. Deep Python required."))
print("Frontend match:  ", rag_agent.match("React, TypeScript, WebGL. Strong CSS. No backend needed."))

# Test salary estimation
print("Salary estimate: ", salary_agent.estimate("AI Engineer", "Anthropic"))

## Autonomous Planning Agent

GPT with tool-calling autonomously decides: **scan → match each job → estimate salaries → pick best → notify**

In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "scan_job_boards",
        "description": "Scan the internet for current tech job listings",
        "parameters": {"type": "object", "properties": {}, "required": [], "additionalProperties": False}
    }},
    {"type": "function", "function": {
        "name": "match_resume",
        "description": "Match a job description against the user's resume, returns 0-100 score",
        "parameters": {"type": "object", "properties": {
            "job_description": {"type": "string", "description": "The job description to match"}
        }, "required": ["job_description"], "additionalProperties": False}
    }},
    {"type": "function", "function": {
        "name": "estimate_salary",
        "description": "Estimate the fair market salary for a job title at a company",
        "parameters": {"type": "object", "properties": {
            "job_title": {"type": "string"}, "company": {"type": "string"}
        }, "required": ["job_title", "company"], "additionalProperties": False}
    }},
    {"type": "function", "function": {
        "name": "notify_user",
        "description": "Send a push notification about the single best job opportunity",
        "parameters": {"type": "object", "properties": {
            "title": {"type": "string"}, "company": {"type": "string"},
            "match_score": {"type": "number"}, "estimated_salary": {"type": "number"},
            "url": {"type": "string"}
        }, "required": ["title", "company", "match_score", "estimated_salary", "url"], "additionalProperties": False}
    }}
]

In [ ]:
class PlanningAgent(Agent):
    """Autonomous orchestrator — GPT decides which tools to call and in what order."""
    name = "Planner"

    SYSTEM = """You are an autonomous job hunting assistant. Use your tools to:
1. Scan for jobs
2. Match each against the user's resume
3. Estimate salary for each
4. Notify about the single best opportunity (highest match + good salary)
Then respond OK."""

    def __init__(self, scanner, rag_agent, salary_agent, messenger):
        self.tool_map = {
            "scan_job_boards": lambda: scanner.scan().model_dump_json(),
            "match_resume": lambda job_description: json.dumps(rag_agent.match(job_description)),
            "estimate_salary": lambda job_title, company: json.dumps(salary_agent.estimate(job_title, company)),
            "notify_user": lambda **kw: messenger.alert(**kw),
        }
        self.scanner = scanner
        self.messenger = messenger
        self.log("Ready")

    def plan(self, use_test_data=False) -> Optional[JobOpportunity]:
        self.log("Starting autonomous job hunt...")
        if use_test_data:
            self.tool_map["scan_job_boards"] = lambda: self.scanner.test_scan().model_dump_json()

        messages = [
            {"role": "system", "content": self.SYSTEM},
            {"role": "user", "content": "Find me the best job. Scan, evaluate each, and notify me about the top one."}
        ]

        best = None
        for step in range(1, 16):
            resp = openai_client.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=TOOLS)
            choice = resp.choices[0]

            if choice.finish_reason != "tool_calls":
                self.log(f"Done in {step} steps: {choice.message.content}")
                break

            msg = choice.message
            self.log(f"Step {step}: {[tc.function.name for tc in msg.tool_calls]}")
            results = []
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                out = self.tool_map[tc.function.name](**args)
                results.append({"role": "tool", "content": out if isinstance(out, str) else json.dumps(out), "tool_call_id": tc.id})
                if tc.function.name == "notify_user":
                    best = JobOpportunity(
                        job=JobListing(title=args["title"], company=args["company"], description="", url=args["url"]),
                        match_score=args["match_score"], estimated_salary=args["estimated_salary"],
                        overall_score=args["match_score"]
                    )
            messages.append(msg)
            messages.extend(results)

        return best

## Run It! (Test Data)

Watch GPT autonomously orchestrate the full pipeline:

In [ ]:
planner = PlanningAgent(scanner, rag_agent, salary_agent, messenger)

result = planner.plan(use_test_data=True)
if result:
    print(f"\nBest match: {result.summary()}")

## Gradio Dashboard

A simple UI with a "Hunt" button, opportunity table, and live logs.

In [ ]:
MEMORY_FILE = "job_memory.json"

def load_memory():
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE) as f:
            return [JobOpportunity(**x) for x in json.load(f)]
    return []

def save_memory(opps):
    with open(MEMORY_FILE, "w") as f:
        json.dump([o.model_dump() for o in opps], f, indent=2)

memory = load_memory()


def table_for(opps):
    return [[o.job.title, o.job.company, f"{o.match_score:.0f}%", f"${o.estimated_salary:,.0f}", o.job.url] for o in opps]


def run_hunt():
    global memory
    p = PlanningAgent(ScannerAgent(), ResumeRAGAgent(SAMPLE_RESUME), SalaryEstimatorAgent(), MessagingAgent())
    result = p.plan(use_test_data=True)  # Change to False for live RSS
    if result:
        memory.append(result)
        save_memory(memory)
    return table_for(memory)


with gr.Blocks(title="Job Opportunity Hunter", theme=gr.themes.Soft()) as ui:
    gr.Markdown("## Job Opportunity Hunter")
    gr.Markdown("Multi-agent system: Scanner → RAG Resume Match → Salary Estimator → Planner → Notify")

    hunt_btn = gr.Button("Hunt for Jobs", variant="primary")

    table = gr.Dataframe(
        headers=["Title", "Company", "Match", "Est. Salary", "URL"],
        value=table_for(memory),
        wrap=True, column_widths=[3, 2, 1, 2, 3], row_count=10, col_count=5,
        max_height=400, label="Discovered Opportunities"
    )

    hunt_btn.click(fn=run_hunt, outputs=[table])

ui.launch(inbrowser=True)

In [ ]:
# Uncomment to run with LIVE RSS feeds instead of test data:
# Change use_test_data=True to use_test_data=False in the run_hunt() function above